# 33 — Final test evaluation: encoder roster

**Protocol.** Each encoder was selected-era trained on `train` and ranked on `dev` in notebooks
11–16. Here every encoder is refit on **train+dev** (49,990 rows) and scored **once** on the
official held-out **test** set (15,395 rows, 505 Negative), pooled across all five languages.

**Where the training ran.** Encoders need a GPU, so the fits execute on Kaggle T4s via the runner:

```bash
python ml/kaggle/runner.py run --models xlmr-base,labse,mmbert,canine-c,sinbert-large,sinhalaberto \
    --epochs 3 --fit-portion train+dev --eval-portion test
python ml/kaggle/runner.py fetch
```

Results land in `ml/reports/runs/*__ev-all__*__test.json`, stamped with the split sha, and are
read below alongside the classical test runs from
[`32_final_test_classical.ipynb`](32_final_test_classical.ipynb).

**Honesty note.** Scoring the whole roster on test (not just the single dev winner) spends test's
role as an unbiased future estimate. The trade is a tighter Negative-F1 read: test has 505 pooled
Negative tickets vs dev's ~340. The Sinhala-only checkpoints are trained on Sinhala alone, so their
pooled cell is not a fair number — read them on the sinhala cell only (per-language files).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, splits, results

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

ENCODERS = {"xlmr-base", "mmbert", "labse", "canine-c", "sinbert-large", "sinhalaberto"}
MONO = {"sinbert-large", "sinhalaberto"}   # Sinhala-only: unfair on the pooled cell
print("split sha:", splits.sha())

split sha: e7b5934392cd


In [2]:
runs = results.load_all()
runs["family"] = runs.get("family", pd.Series(["classical"] * len(runs))).fillna("classical")
runs.loc[runs.model.isin(ENCODERS), "family"] = "encoder"

test = runs[(runs.task == "sentiment") & (runs.eval_portion == "test") & (runs.eval_lang == "all")].copy()
print(f"{len(test)} pooled test runs on split {splits.sha()}")
if test.empty:
    raise SystemExit("no pooled test runs yet -- run the Kaggle test batch + classical notebook first")

cols = [c for c in ["model", "family", "arm", "headline", "accuracy",
                    "negative_precision", "negative_recall", "author"] if c in test.columns]
board = (test.sort_values("headline", ascending=False)
             .drop_duplicates("model")[cols]
             .rename(columns={"headline": "test_negF1"}))
display(board)
print("best on test:", board.iloc[0]["model"], f"{board.iloc[0]['test_negF1']:.4f}")

8 pooled test runs on split e7b5934392cd


,model,family,arm,test_negF1,accuracy,negative_precision,negative_recall,author
188,labse,encoder,class_weight,0.5664,0.9699,0.5363,0.6000,kaggle
373,xlmr-base,encoder,class_weight,0.5396,0.9641,0.4655,0.6416,kaggle
195,mmbert,encoder,class_weight,0.5321,0.9682,0.5148,0.5505,kaggle
186,canine-c,encoder,class_weight,0.4702,0.9578,0.4000,0.5703,kaggle
329,tfidf-svm,classical,class_weight,0.4635,0.9517,0.3648,0.6356,final-test-classical
243,tfidf-logreg,classical,ros,0.4498,0.9530,0.3650,0.5861,final-test-classical
199,sinhalaberto,encoder,class_weight,0.1296,0.6309,0.0702,0.8376,kaggle
197,sinbert-large,encoder,class_weight,0.1182,0.7530,0.0670,0.5050,kaggle


best on test: labse 0.5664


In [3]:
# dev vs test, pooled, side by side -- how much each model gives back opening test
dev = runs[(runs.task == "sentiment") & (runs.eval_portion == "dev") & (runs.eval_lang == "all")]
dev = dev.sort_values("headline", ascending=False).drop_duplicates("model")[["model", "headline"]]
dev = dev.rename(columns={"headline": "dev_negF1"})
tst = board.rename(columns={"test_negF1": "test_negF1"})[["model", "family", "test_negF1"]]
cmp = tst.merge(dev, on="model", how="left")
cmp["drop"] = cmp["dev_negF1"] - cmp["test_negF1"]
cmp = cmp.sort_values("test_negF1", ascending=False)
display(cmp)

fair = cmp[~cmp.model.isin(MONO)]
print(f"\nBest test Negative-F1 (excl. Sinhala-only): {fair.iloc[0]['model']} "
      f"{fair.iloc[0]['test_negF1']:.4f}")
cbest = fair[fair.family == 'classical']['test_negF1'].max()
ebest = fair[fair.family == 'encoder']['test_negF1'].max()
print(f"classical best test {cbest:.4f}   encoder best test {ebest:.4f}   "
      f"gap {ebest - cbest:+.4f}")

,model,family,test_negF1,dev_negF1,drop
0,labse,encoder,0.5664,0.6334,0.0671
1,xlmr-base,encoder,0.5396,0.5012,-0.0383
2,mmbert,encoder,0.5321,0.6203,0.0882
3,canine-c,encoder,0.4702,0.5323,0.0621
4,tfidf-svm,classical,0.4635,0.5954,0.1319
5,tfidf-logreg,classical,0.4498,0.5720,0.1222
6,sinhalaberto,encoder,0.1296,0.1730,0.0434
7,sinbert-large,encoder,0.1182,0.1238,0.0055



Best test Negative-F1 (excl. Sinhala-only): labse 0.5664
classical best test 0.4635   encoder best test 0.5664   gap +0.1028


## Reading this honestly

1. **Rank on the CI, not the point estimate.** Even with 505 Negative tickets the test Negative-F1
   CI is roughly ±0.06. A gap smaller than that is not a real ordering — do not crown a winner on a
   hairline lead.
2. **Dev→test drop is expected and large** (~0.15 for the classical champion). Any encoder that led
   on dev will likely give back ground here too; that is the point of measuring it.
3. **Sinhala-only models** (sinbert-large, sinhalaberto) look terrible on the pooled cell because
   they only speak Sinhala — judge them on the sinhala per-language cell, not this table.
4. The `gap` line above is the bottom line: **did any encoder clear the classical champion on test
   by more than the CI?** If not, the simpler classical model is what ships.